# D1 - Capacità fiscale IRPEF 2019-2023

Notebook di supporto alla Discussion IRPEF su `dataciviclab`.

**Perimetro**
- anni: `2019-2023`
- livelli: `regione` e `comune`
- base dati: `irpef_capacita_fiscale_multi_anno.parquet`
- focus: differenze territoriali, ranking e anomalie semplici

**Nota metodologica**
- qui non uso popolazione
- il notebook lavora su contribuenti, reddito imponibile e medie per contribuente
- le letture comunali vanno tenute prudenti: la Discussion resta prima di tutto un confronto territoriale esplorativo


In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd


pd.options.display.float_format = '{:,.2f}'.format


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'preanalysis').exists() and (candidate / '_out').exists():
            return candidate
    raise RuntimeError('Repo root non trovato')


REPO_ROOT = find_repo_root(Path.cwd())
DATA_PATH = REPO_ROOT / '_out' / 'data' / 'cross' / 'irpef_comunale_2019_2023' / 'irpef_capacita_fiscale_multi_anno.parquet'
ANNI = [2019, 2020, 2021, 2022, 2023]
GRANDI_COMUNI = ['ROMA', 'MILANO', 'TORINO', 'GENOVA', 'NAPOLI', 'BOLOGNA', 'PALERMO', 'FIRENZE', 'BARI']

con = duckdb.connect()
str(DATA_PATH)


## 1. Copertura del parquet multi-anno

Verifico che il file cross-year copra tutti gli anni e i due livelli territoriali attesi.


In [ ]:
copertura = con.execute(
    """
    select
        anno_imposta,
        livello_territoriale,
        count(*) as righe,
        count(distinct territorio) as territori_distinti
    from read_parquet($path)
    group by 1, 2
    order by 1, 2
    """,
    {'path': str(DATA_PATH)},
).fetchdf()

copertura


## 2. Regioni: chi pesa di più e come evolve nel tempo

Parto dal livello regionale, che è il più leggibile per una prima risposta pubblica.


In [ ]:
regioni_2023 = con.execute(
    """
    select
        territorio as regione,
        reddito_imponibile_totale_eur,
        numero_contribuenti,
        reddito_imponibile_medio_per_contribuente_eur,
        rank_nazionale_reddito_imponibile
    from read_parquet($path)
    where livello_territoriale = 'regione'
      and anno_imposta = 2023
    order by reddito_imponibile_totale_eur desc
    limit 10
    """,
    {'path': str(DATA_PATH)},
).fetchdf()

regioni_2023['reddito_imponibile_totale_mld'] = (regioni_2023['reddito_imponibile_totale_eur'] / 1e9).round(2)
regioni_2023


In [ ]:
trend_regioni = con.execute(
    """
    select
        anno_imposta,
        territorio as regione,
        reddito_imponibile_totale_eur
    from read_parquet($path)
    where livello_territoriale = 'regione'
      and territorio in ('Lombardia', 'Lazio', 'Veneto', 'Emilia Romagna', 'Piemonte')
    order by regione, anno_imposta
    """,
    {'path': str(DATA_PATH)},
).fetchdf()

fig, ax = plt.subplots(figsize=(10, 6))
for regione, df_regione in trend_regioni.groupby('regione'):
    ax.plot(df_regione['anno_imposta'], df_regione['reddito_imponibile_totale_eur'], marker='o', label=regione)

ax.set_title('Reddito imponibile totale: prime regioni 2019-2023')
ax.set_xlabel('Anno')
ax.set_ylabel('Reddito imponibile totale (euro)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.0f} mld'))
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 3. Comuni: profilo delle grandi città

Qui tengo i principali comuni emersi dal ranking, per collegare meglio la lettura regionale a casi territoriali concreti.


In [ ]:
comuni_2023 = con.execute(
    """
    select
        territorio as comune,
        regione,
        numero_contribuenti,
        reddito_imponibile_totale_eur,
        reddito_imponibile_medio_per_contribuente_eur,
        rank_nazionale_reddito_imponibile,
        rank_regionale_reddito_imponibile,
        quota_reddito_imponibile_su_regione
    from read_parquet($path)
    where livello_territoriale = 'comune'
      and anno_imposta = 2023
      and territorio in $grandi_comuni
    order by reddito_imponibile_totale_eur desc
    """,
    {'path': str(DATA_PATH), 'grandi_comuni': GRANDI_COMUNI},
).fetchdf()

comuni_2023['reddito_imponibile_totale_mld'] = (comuni_2023['reddito_imponibile_totale_eur'] / 1e9).round(2)
comuni_2023


## 4. Anomalie semplici: imponibile totale vs numero di contribuenti

Questa vista serve a capire se emergono comuni con combinazioni non banali tra platea e imponibile medio.


In [ ]:
anomalie_2023 = con.execute(
    """
    select
        territorio as comune,
        regione,
        numero_contribuenti,
        reddito_imponibile_totale_eur,
        reddito_imponibile_medio_per_contribuente_eur
    from read_parquet($path)
    where livello_territoriale = 'comune'
      and anno_imposta = 2023
      and numero_contribuenti >= 50000
    order by reddito_imponibile_medio_per_contribuente_eur desc
    limit 20
    """,
    {'path': str(DATA_PATH)},
).fetchdf()

anomalie_2023['reddito_imponibile_totale_mld'] = (anomalie_2023['reddito_imponibile_totale_eur'] / 1e9).round(2)
anomalie_2023


## Note operative

- il notebook usa il parquet cross-year nativo prodotto dal layer `cross_year`
- la lettura pubblica va tenuta stretta: regioni, grandi comuni, anomalie semplici
- senza popolazione, qui non sto ancora leggendo capacità fiscale pro capite
- un buon primo commento sotto la Discussion può appoggiarsi a regioni top, grandi comuni selezionati e 2-3 anomalie difendibili
